In [3]:
import pandas as pd

print("tải thành công superstore_final.csv...")
df = pd.read_csv('../data/processed/superstore_final.csv')

tải thành công superstore_final.csv...


In [6]:
# 1. DIM CUSTOMER
dim_customer = df[['customer_id', 'customer_name', 'segment', 'recency', 'frequency', 'monetary', 'rfm_score', 'customer_segment']].drop_duplicates()
dim_customer.to_csv('../data/processed/star_schema/dim_customer.csv', index=False)
print(f"Đã tạo: {dim_customer.shape[0]} khách hàng.")

Đã tạo: 4873 khách hàng.


In [8]:
# 2. DIM PRODUCT
dim_product = df[['product_id', 'category', 'sub_category', 'product_name']].drop_duplicates()
dim_product.to_csv('../data/processed/star_schema/dim_product.csv', index=False)
print(f"Đã tạo dim_product: {dim_product.shape[0]} sản phẩm.")

Đã tạo dim_product: 10768 sản phẩm.


In [9]:
# 3. DIM LOCATION
# Tạo surrogate key (Location_ID) cho Dimension này
dim_location = df[['market', 'region', 'country', 'state', 'city']].drop_duplicates().reset_index(drop=True)
dim_location['location_id'] = dim_location.index + 1
dim_location.to_csv('../data/processed/star_schema/dim_location.csv', index=False)
print(f"Đã tạo dim_location: {dim_location.shape[0]} địa điểm.")

Đã tạo dim_location: 3819 địa điểm.


In [10]:
# 4. DIM TIME
# Tạo bảng Lịch từ chuỗi ngày order_date
dim_time = pd.DataFrame({'date': pd.to_datetime(df['order_date'].unique())}).sort_values('date')
dim_time['year'] = dim_time['date'].dt.year
dim_time['month'] = dim_time['date'].dt.month
dim_time['quarter'] = "Q" + dim_time['date'].dt.quarter.astype(str)
dim_time['day_of_week'] = dim_time['date'].dt.day_name()
dim_time.to_csv('../data/processed/star_schema/dim_time.csv', index=False)
print(f"Đã tạo dim_time: {dim_time.shape[0]} ngày giao dịch.")

Đã tạo dim_time: 1430 ngày giao dịch.


In [ ]:
# 5. FACT SALES
# Map Location_ID ngược lại vào Fact Table
fact_sales = df.merge(dim_location, on=['market', 'region', 'country', 'state', 'city'], how='left')

# Chỉ giữ lại các ID khóa (Foreign Keys) và Metrics (Measures)
fact_cols = [
    'order_id', 'customer_id', 'product_id', 'location_id', 'order_date', 'ship_date', 'ship_mode', 'order_priority',
    'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 
    'delivery_days', 'is_late_delivery', 'profit_margin', 'is_loss_maker', 'discount_level'
]
fact_sales = fact_sales[fact_cols]
fact_sales.to_csv('../data/processed/star_schema/fact_sales.csv', index=False)
print(f"Đã tạo fact_sales: {fact_sales.shape[0]} dòng giao dịch.")

Đã tạo fact_sales: 51252 dòng giao dịch.


: 